<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Predic%C8%9Bia_pre%C8%9Bului_aurului.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor

In [30]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [31]:
df_train

,ID,date,sp500 open,sp500 high,sp500 low,sp500 close,sp500 volume,sp500 high-low,nasdaq open,nasdaq high,nasdaq low,nasdaq close,nasdaq volume,nasdaq high-low,us_rates_%,CPI,usd_chf,eur_usd,GDP,silver open,silver high,silver low,silver close,silver volume,silver high-low,oil open,oil high,oil low,oil close,oil volume,oil high-low,platinum open,platinum high,platinum low,platinum close,platinum volume,platinum high-low,palladium open,palladium high,palladium low,palladium close,palladium volume,palladium high-low,gold open,gold high,gold low,gold close,gold volume
0,R00210,2010-11-11,121.05,121.8200,120.68,121.64,157659616.0,1.1400,52.90,53.4800,52.660,53.385,85163600.0,0.8200,NaN,NaN,NaN,NaN,NaN,27.40,27.6698,26.8400,27.62,372618.0,0.8298,37.90,38.0400,37.76,37.84,8849109.0,0.2800,174.45,174.7000,173.2600,174.64,39800.0,1.4400,71.30,71.4800,69.4500,71.0700,279282.0,2.0300,137.6200,137.7500,136.4500,137.66,15380502.0
1,R03365,2023-05-26,415.33,420.7700,415.25,420.02,93829975.0,5.5200,340.76,349.2450,340.660,348.400,63006952.0,8.5850,NaN,NaN,0.9060,1.0713,NaN,22.21,22.3850,22.0822,22.35,389291.0,0.3028,64.90,64.9900,64.36,64.80,2145281.0,0.6300,94.79,95.6600,94.4500,94.45,83380.0,1.2100,132.36,133.6100,131.3700,131.3700,20046.0,2.2400,181.0100,181.3000,180.0900,180.92,5823674.0
2,R03382,2023-06-22,433.95,436.6200,433.60,436.51,70637175.0,3.0200,360.63,366.3300,360.220,366.170,47603043.0,6.1100,NaN,NaN,0.8967,1.0953,NaN,21.56,21.6400,21.3836,21.44,409222.0,0.2564,63.15,63.6500,62.14,62.52,3882134.0,1.5100,86.63,86.8169,85.3300,85.33,164282.0,1.4869,122.23,122.2300,116.9100,118.8600,74797.0,5.3200,178.3600,178.9900,177.6300,177.71,7948565.0
3,R01294,2015-03-06,210.46,210.4600,207.10,207.50,188127982.0,3.3600,108.50,108.7100,107.140,107.410,30990366.0,1.5700,NaN,NaN,0.9846,1.0855,NaN,15.65,15.6800,15.5200,15.63,141738.0,0.1600,18.41,18.5500,17.97,18.24,27455956.0,0.5800,112.97,112.9700,112.2500,112.35,64345.0,0.7200,79.20,79.7300,79.0500,79.2800,60988.0,0.6800,113.1700,113.2600,111.7000,111.86,11190846.0
4,R02502,2019-12-20,320.46,321.9700,319.39,320.73,149214080.0,2.5800,211.86,212.5200,211.270,211.710,27690934.0,1.2500,NaN,NaN,0.9824,1.1076,NaN,16.65,16.7200,16.5700,16.64,92711.0,0.1500,12.72,12.7200,12.56,12.63,11434745.0,0.1600,88.25,88.3100,85.6471,85.81,65233.0,2.6629,182.57,182.5700,172.1033,173.7900,88426.0,10.4667,139.3705,139.5200,138.9800,139.52,4446611.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2970,R01131,2014-07-14,197.61,197.8600,197.44,197.60,58657919.0,0.4200,95.70,96.0550,95.535,95.840,19579000.0,0.5200,NaN,NaN,0.8918,1.3620,NaN,20.60,20.6500,20.5400,20.62,98231.0,0.1100,37.06,37.3500,36.96,37.32,6182303.0,0.3900,145.38,145.8000,144.9600,145.18,56000.0,0.8400,84.60,84.9000,84.4134,84.6000,40499.0,0.4866,125.5000,126.1000,125.4500,125.72,11469917.0
2971,R01295,2015-03-09,207.74,208.7900,207.55,208.36,89818893.0,1.2400,107.63,107.9857,107.190,107.720,26237101.0,0.7957,NaN,NaN,0.9873,1.0846,NaN,15.58,15.6100,15.5300,15.54,91620.0,0.0800,18.18,18.6350,18.14,18.35,19728959.0,0.4950,112.05,112.0500,111.2500,111.39,42945.0,0.8000,79.75,80.3899,79.6510,79.8300,18585.0,0.7389,112.4200,112.6200,111.9500,111.97,5534011.0
2972,R00861,2013-06-17,164.29,165.2200,163.22,164.44,136114269.0,2.0000,72.88,73.4000,72.600,72.980,32441900.0,0.8000,NaN,NaN,0.9254,1.3341,NaN,21.62,21.6900,21.5501,21.61,73387.0,0.1399,34.86,34.9400,34.60,34.76,4296019.0,0.3400,141.40,141.5800,140.3500,140.51,65000.0,1.2300,70.57,70.6399,69.7800,69.8100,220029.0,0.8599,133.8900,134.0599,133.5201,133.77,4072212.0
2973,R03508,2023-12-20,473.96,475.8950,467.82,468.26,102920959.0,8.0750,408.35,410.4699,402.900,403.080,54042443.0,7.5699,NaN,NaN,0.8615,1.0957,NaN,23.30,23.3300,23.0500,23.05,527660.0,0.2800,70.15,70.3001,68.66,68.79,5252060.0,1.6401,88.33,89.4800,88.2600,88.43,280711.0,1.2200,112.41,112.7800,10

In [32]:
df_train.dtypes

,0
ID,object
date,object
sp500 open,float64
sp500 high,float64
sp500 low,float64
sp500 close,float64
sp500 volume,float64
sp500 high-low,float64
nasdaq open,float64
nasdaq high,float64


In [33]:


def impute_columns(df_train, df_test, columns, strategy):
    imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    df_train[columns] = imputer.fit_transform(df_train[columns])
    df_test[columns] = imputer.transform(df_test[columns])

def scale_columns(df_train, df_test, columns):
    scaler = MinMaxScaler()
    df_train[columns] = scaler.fit_transform(df_train[columns])
    df_test[columns] = scaler.transform(df_test[columns])

def process(df_train, df_test):

    numeric_columns = [
        'sp500 open', 'sp500 high', 'sp500 low', 'sp500 close', 'sp500 volume', 'sp500 high-low',
        'nasdaq open', 'nasdaq high', 'nasdaq low', 'nasdaq close', 'nasdaq volume', 'nasdaq high-low',
        'us_rates_%', 'CPI', 'usd_chf', 'eur_usd', 'GDP',
        'silver open', 'silver high', 'silver low', 'silver close', 'silver volume', 'silver high-low',
        'oil open', 'oil high', 'oil low', 'oil close', 'oil volume', 'oil high-low',
        'platinum open', 'platinum high', 'platinum low', 'platinum close', 'platinum volume', 'platinum high-low',
        'palladium open', 'palladium high', 'palladium low', 'palladium close', 'palladium volume', 'palladium high-low',
        'gold open', 'gold high', 'gold low', 'gold volume'
    ]
    impute_columns(df_train, df_test, numeric_columns, 'mean')
    scale_columns(df_train, df_test, numeric_columns)

    drop_list = ['date']
    df_train = df_train.drop(drop_list, axis=1)
    df_test = df_test.drop(drop_list, axis=1)


In [34]:
process(df_train, df_test)

In [35]:
X = df_train.drop(['gold close'], axis=1)
y = df_train['gold close']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"Train: {root_mean_squared_error(y_train, y_train_pred)}, Test: {root_mean_squared_error(y_test, y_test_pred)}")

In [37]:
# Convert ID to numeric and drop date directly in the split datasets
X_train['ID'] = X_train['ID'].apply(lambda x: int(''.join(filter(str.isdigit, str(x)))))
X_test['ID'] = X_test['ID'].apply(lambda x: int(''.join(filter(str.isdigit, str(x)))))

X_train = X_train.drop(columns=['date'])
X_test = X_test.drop(columns=['date'])

lr = LinearRegression()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 0.28230942802224995, Test: 0.2752580316771128


In [38]:
def solve(model):
    y_pred = model.predict(df_test)

    output = pd.DataFrame({'ID': df_test['ID'].apply(lambda x: f'R{x}'), 'gold close': y_pred})
    output.to_csv('submission.csv', index=False)

In [41]:
def solve(model):
    # Keep a copy of the original IDs to restore later
    original_ids = df_test['ID'].copy()

    # Prepare the test features to match the training feature structure exactly
    X_test_final = df_test.copy()

    # 1. Convert ID to numeric integer representation
    X_test_final['ID'] = X_test_final['ID'].apply(lambda x: int(''.join(filter(str.isdigit, str(x)))))

    # 2. Drop the 'date' column
    X_test_final = X_test_final.drop(columns=['date'])

    y_pred = model.predict(X_test_final)

    output = pd.DataFrame({'ID': original_ids, 'gold close': y_pred.round(2)})
    output.to_csv('submission.csv', index=False)

solve(lr)